# DiffusionNFT 代码实现详解

> **论文**: *DiffusionNFT: Online Diffusion Reinforcement with Forward Process*  
> **会议**: ICLR 2026 Oral (NVIDIA)

## 核心思想

DiffusionNFT 将在线强化学习搬到扩散模型的**前向加噪过程（Forward Process）**中，用纯粹的有监督流匹配（Flow Matching）统一了正负反馈。

**与传统 Diffusion RL（如 FlowGRPO）的关键区别**：
- FlowGRPO：在反向采样过程上做策略梯度 → 必须用一阶 SDE 采样器、存储完整轨迹、显存爆炸
- **DiffusionNFT**：在前向加噪过程上做流匹配 → 可用任意高阶 ODE 采样器、只需干净图像 $x_0$、显存骤降

**本 Demo 覆盖的核心要点**：
1. **Flow Matching 前向过程**：$x_t = (1-t) \cdot x_0 + t \cdot \epsilon$
2. **隐式速度导向**：$v^+ = \beta v_\theta + (1-\beta) v_{\text{old}}$，$v^- = (1+\beta) v_{\text{old}} - \beta v_\theta$
3. **奖励加权策略损失**：$\mathcal{L} = r \cdot \mathcal{L}_{\text{pos}} / \beta + (1-r) \cdot \mathcal{L}_{\text{neg}} / \beta$
4. **Old Policy 的 EMA 更新**
5. **Per-Prompt Advantage 归一化**

下面逐模块拆解代码实现 👇

## 导入依赖与超参数设置

导入 PyTorch 相关库并设置所有超参数，对应论文代码中的 `config/nft.py` 配置文件。

关键超参数说明：
- `BETA`（$\beta$）：**正/负速度场混合系数**，DiffusionNFT 的核心超参。控制隐式正策略和隐式负策略的拉扯强度
- `BETA_KL`：KL 正则化系数，防止策略偏离 reference model 太远
- `ADV_CLIP_MAX`：advantage 截断范围，用于奖励归一化
- `DECAY_TYPE`：old policy 的 EMA 更新策略

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

torch.manual_seed(42)    # 固定随机种子, 保证结果可复现
device = "cpu"

# ---- 超参数 (对应 config/nft.py 中的 sd3_geneval 配置) ----
BETA = 1.0              # 正/负速度场混合系数 β (论文核心超参)
BETA_KL = 1e-4          # KL 正则化系数 β_kl
ADV_CLIP_MAX = 5.0      # advantage 截断范围
LR = 3e-4               # AdamW 学习率
LATENT_DIM = 4           # latent 通道数 (SD3 用 16, 这里简化为 4)
LATENT_HW = 8            # latent 空间分辨率 (SD3 用 64, 这里简化为 8)
BATCH_SIZE = 4           # batch size (每个 prompt 生成的样本数)
NUM_TIMESTEPS = 4        # 每个样本训练的时间步数 (原代码用 1000 中采样)
DECAY_TYPE = 1           # old policy EMA 衰减策略 (0=纯on-policy, 1=温和, 2=延迟)

## 简易速度场网络（代替 SD3 的 Transformer）

在真实的 DiffusionNFT 中，速度场预测网络是 SD3 的 MMDiT Transformer（数十亿参数）。本 demo 用一个简单的 3 层 MLP 代替。

**输入**：带噪 latent $x_t$、时间步 $t$、条件嵌入 $c$（文本 prompt 的编码）

**输出**：速度场 $v_\theta(x_t, t, c)$，与 $x_t$ 同维度

在 **Rectified Flow Matching** 框架中，速度场定义了从噪声到数据的"流"的方向。通过积分速度场，可以从噪声恢复出干净图像：

$$\hat{x}_0 = x_t - t \cdot v_\theta(x_t, t, c)$$

In [ ]:
class SimpleVelocityNet(nn.Module):
    """简单的速度场网络 v_θ(x_t, t, c)
    
    在 SD3 中这是一个巨大的 MMDiT Transformer, 这里用 MLP 代替.
    
    输入:
        x_t:  [B, C, H, W] 带噪 latent
        t:    [B] 时间步 (归一化到 [0,1])
        cond: [B, cond_dim] 条件嵌入 (代替 prompt_embeds)
    输出:
        v: [B, C, H, W] 速度场预测 (与 x_t 同维度)
    """
    def __init__(self, dim=LATENT_DIM * LATENT_HW * LATENT_HW, hidden=128, cond_dim=32):
        super().__init__()
        self.dim = dim
        self.hw = LATENT_HW
        self.channels = LATENT_DIM
        # 简单的 3 层 MLP: 拼接 [x_t 展平, t, cond] → hidden → hidden → 输出
        self.net = nn.Sequential(
            nn.Linear(dim + 1 + cond_dim, hidden),    # 输入: 展平的 x_t + 时间步 + 条件
            nn.SiLU(),                                  # 激活函数
            nn.Linear(hidden, hidden),                  # 隐藏层
            nn.SiLU(),
            nn.Linear(hidden, dim),                     # 输出层
        )

    def forward(self, x_t, t, cond):
        B = x_t.shape[0]
        x_flat = x_t.view(B, -1)             # [B, C*H*W] 展平空间维度
        t_expand = t.view(B, 1)              # [B, 1] 时间步
        inp = torch.cat([x_flat, t_expand, cond], dim=-1)    # 拼接所有输入
        v = self.net(inp)                     # MLP 前向传播
        return v.view(B, self.channels, self.hw, self.hw)    # 恢复为 [B, C, H, W]

## 模拟奖励函数

在真实的 DiffusionNFT 训练中，需要使用外部评价模型对生成的图像打分：
- **GenEval**：评测生成图像的组合理解能力
- **PickScore / HPSv2**：人类偏好评分
- **CLIPScore**：图文匹配度

本 demo 用一个简单的模拟奖励函数代替：$r = -\|x\|^2 + \text{noise}$，L2 范数越小的样本得分越高。

In [ ]:
def mock_reward_fn(images):
    """模拟奖励函数 (代替 GenEval/PickScore/CLIPScore 等)
    
    实际项目中会调用外部评价模型:
    - GenEval: 评测生成图像的组合理解能力
    - PickScore/HPSv2: 人类偏好评分
    - CLIPScore: 图文匹配度
    
    这里用一个简单的图像质量指标代替: 基于 latent 的 L2 范数 + 随机噪声
    L2 范数越小 → 图像越 "干净" → 得分越高
    """
    scores = -images.pow(2).mean(dim=[1, 2, 3]) + 0.5 * torch.randn(images.shape[0])
    return scores    # [B] 每个样本一个标量奖励值

## Per-Prompt 奖励统计追踪器

按 prompt 统计奖励的均值和标准差，计算 advantage（对应原代码 `stat_tracking.py`）：

$$\text{advantage} = \frac{\text{reward} - \text{per\_prompt\_mean}}{\text{std}}$$

- **Per-Prompt 均值**：每个 prompt 独立维护历史奖励均值，作为该 prompt 的 baseline
- **全局标准差**（`global_std=True`）：使用当前 batch 的全局标准差，这与 GRPO 的组内归一化风格一致
- 累积历史奖励：随着训练推进，统计越来越准确

In [ ]:
class PerPromptStatTracker:
    """按 prompt 统计奖励的均值和标准差, 计算 advantage
    (对应原代码 flow_grpo/stat_tracking.py)
    
    核心公式:
        advantage = (reward - per_prompt_mean) / std
    
    参数:
        global_std: 是否使用全局标准差 (True=GRPO 风格, False=per-prompt std)
    """
    def __init__(self, global_std=True):
        self.global_std = global_std
        self.stats = {}    # 按 prompt 存储历史奖励

    def update(self, prompts, rewards):
        """更新统计并返回 advantage
        
        参数:
            prompts: prompt 标识列表 (如文本字符串)
            rewards: 对应的奖励数组
        返回:
            advantages: 归一化后的优势值
        """
        prompts = np.array(prompts)
        rewards = np.array(rewards, dtype=np.float64)
        unique = np.unique(prompts)
        advantages = np.empty_like(rewards)

        # 第一步: 将新奖励追加到每个 prompt 的历史统计中
        for prompt in unique:
            if prompt not in self.stats:
                self.stats[prompt] = []
            prompt_rewards = rewards[prompts == prompt]
            self.stats[prompt].extend(prompt_rewards)

        # 第二步: 对每个 prompt 组计算 advantage
        for prompt in unique:
            arr = np.stack(self.stats[prompt])
            mean = np.mean(arr)                         # 该 prompt 的历史均值

            if self.global_std:
                std = np.std(rewards) + 1e-4            # 全局标准差 (GRPO 风格)
            else:
                std = np.std(arr) + 1e-4                # 组内标准差

            # advantage = (reward - mean) / std
            advantages[prompts == prompt] = (rewards[prompts == prompt] - mean) / std

        return advantages

## Old Policy 衰减函数（EMA 软更新）

控制 old policy 的 EMA 更新速率：

$$\theta^{\text{old}} = \eta \cdot \theta^{\text{old}} + (1 - \eta) \cdot \theta$$

- **`decay_type=0`**：$\eta=0$，old policy 每步完全替换为当前策略（纯 on-policy，容易训练崩溃）
- **`decay_type=1`**：$\eta$ 从 0 线性增长到 0.5，温和增长（推荐）
- **`decay_type=2`**：前 75 步 $\eta=0$，之后快速增长到 0.999

**设计动机**：平衡在线学习的速度（$\eta$ 小）与稳定性（$\eta$ 大）。如果 old policy 更新太快，新旧策略偏差过大会导致训练不稳定。

In [ ]:
def return_decay(step, decay_type):
    """控制 old policy 的 EMA 衰减速率 (对应 train_nft_sd3.py L169)
    
    EMA 更新公式: θ_old = decay * θ_old + (1 - decay) * θ_current
    - decay=0: old policy 每步完全替换为当前策略 (纯 on-policy, 不稳定)
    - decay→1: old policy 几乎不更新 (接近离线, 稳定但响应慢)
    
    参数:
        step: 当前训练步数
        decay_type: 衰减策略类型
            0 = 固定为 0 (纯 on-policy)
            1 = 从 0 线性增长到 0.5 (速率 0.001/步)
            2 = 前 75 步为 0, 之后线性增长到 0.999 (速率 0.0075/步)
    """
    if decay_type == 0:
        flat, uprate, uphold = 0, 0.0, 0.0       # 纯 on-policy
    elif decay_type == 1:
        flat, uprate, uphold = 0, 0.001, 0.5      # 温和增长
    elif decay_type == 2:
        flat, uprate, uphold = 75, 0.0075, 0.999  # 延迟启动, 快速增长
    else:
        raise ValueError(f"Unknown decay_type: {decay_type}")

    if step < flat:
        return 0.0       # 在平坦期内, 不做 EMA (完全替换)
    else:
        decay = (step - flat) * uprate
        return min(decay, uphold)    # 线性增长, 但不超过上限

## DiffusionNFT 核心损失函数

这是整篇论文最核心的代码实现，对应论文的 **Theorem 3.2（隐式参数化与双分支联合优化）**。

损失函数的计算流程分为 7 步：

| 步骤 | 公式 | 说明 |
|------|------|------|
| 1 | $r = \text{clamp}(\frac{\text{adv}}{2 \cdot \text{max}} + 0.5, 0, 1)$ | 优势归一化为"最优性概率" |
| 2 | $v^+ = \beta v_\theta + (1-\beta) v_{\text{old}}$ | 正样本速度场（隐式正策略外壳） |
| 2 | $v^- = (1+\beta) v_{\text{old}} - \beta v_\theta$ | 隐式负样本速度场（隐式负策略外壳） |
| 3 | $\hat{x}_0 = x_t - t \cdot v$ | 从速度场恢复 $x_0$ 预测 |
| 4 | $w = \text{mean}(\|\hat{x}_0 - x_0\|).\text{detach}()$ | 自适应加权（免调参） |
| 5 | $\mathcal{L}_{\text{policy}} = r \cdot \mathcal{L}_{\text{pos}} / \beta + (1-r) \cdot \mathcal{L}_{\text{neg}} / \beta$ | 奖励加权的策略损失 |
| 6 | $\mathcal{L}_{\text{KL}} = \|v_\theta - v_{\text{ref}}\|^2$ | KL 正则化 |
| 7 | $\mathcal{L} = \mathcal{L}_{\text{policy}} + \beta_{\text{kl}} \cdot \mathcal{L}_{\text{KL}}$ | 总损失 |

**关键洞察**：只用一个网络 $v_\theta$，通过代数变换构造出正/负两个虚拟速度场，实现了"单模型演双角"的隐式参数化。

In [ ]:
def diffusion_nft_loss(
    v_theta,        # 当前策略的速度场预测 v_θ(x_t, t)
    v_old,          # old policy 的速度场预测 v_old(x_t, t)
    v_ref,          # reference model 的速度场预测 v_ref(x_t, t)
    x0,             # 干净 latent x_0
    x_t,            # 加噪 latent x_t
    t_expanded,     # 时间步 [B, 1, 1, 1]
    advantages,     # 归一化后的 advantage [B]
    beta=BETA,      # 正/负速度场混合系数 β
    beta_kl=BETA_KL,# KL 正则化系数
    adv_clip_max=ADV_CLIP_MAX,
):
    """DiffusionNFT 核心损失函数 (对应 train_nft_sd3.py 第 909-980 行)"""

    # ===== Step 1: 优势归一化到 [0, 1] =====
    # 将 advantage 映射为 "最优性概率" r ∈ [0, 1]
    # r→1 表示高奖励(正样本), r→0 表示低奖励(负样本)
    # 公式: r = clamp(advantage / (2 * adv_max) + 0.5, 0, 1)
    advantages_clip = torch.clamp(advantages, -adv_clip_max, adv_clip_max)
    normalized_advantages = (advantages_clip / adv_clip_max) / 2.0 + 0.5
    r = torch.clamp(normalized_advantages, 0, 1)    # [B]

    # ===== Step 2: 构造正样本和隐式负样本的速度场 =====
    # 正样本速度场: v_pos = β * v_θ + (1-β) * v_old
    #   β=1 时退化为 v_θ, β→0 时退化为 v_old
    #   v_old 用 detach() 截断梯度, 只有 v_theta 接收梯度
    positive_prediction = beta * v_theta + (1 - beta) * v_old.detach()

    # 隐式负样本速度场: v_neg = (1+β) * v_old - β * v_θ
    #   这是关于 v_old 的"镜像外推", 方向远离 v_theta
    #   当 v_theta 靠近好的方向时, v_neg 自动被推向差的方向
    implicit_negative_prediction = (1.0 + beta) * v_old.detach() - beta * v_theta

    # ===== Step 3: 从速度场恢复 x_0 预测 =====
    # Flow Matching 逆公式: x̂_0 = x_t - t * v
    x0_pred_pos = x_t - t_expanded * positive_prediction      # 正样本 x_0 预测
    x0_pred_neg = x_t - t_expanded * implicit_negative_prediction  # 负样本 x_0 预测

    # ===== Step 4: 自适应加权 MSE =====
    # 正样本损失: 先用 |x̂_0 - x_0| 的均值作为自适应权重, 再做加权 MSE
    # 这种自适应加权免去了手动调节不同时间步 t 的权重函数 w(t)
    with torch.no_grad():
        weight_pos = (
            torch.abs(x0_pred_pos - x0)
            .mean(dim=(1, 2, 3), keepdim=True)    # 对 C,H,W 维度求均值
            .clamp(min=1e-5)                       # 防止除零
        )
    positive_loss = ((x0_pred_pos - x0) ** 2 / weight_pos).mean(dim=(1, 2, 3))    # [B]

    # 负样本损失: 同样的自适应加权方式
    with torch.no_grad():
        weight_neg = (
            torch.abs(x0_pred_neg - x0)
            .mean(dim=(1, 2, 3), keepdim=True)
            .clamp(min=1e-5)
        )
    negative_loss = ((x0_pred_neg - x0) ** 2 / weight_neg).mean(dim=(1, 2, 3))    # [B]

    # ===== Step 5: 奖励加权策略损失 =====
    # L_policy = r * L_pos / β + (1-r) * L_neg / β
    # r→1 (高奖励): 主要优化正样本损失 → 让模型靠近好图的速度场
    # r→0 (低奖励): 主要优化负样本损失 → 让模型远离差图的速度场
    policy_loss_per_sample = r * positive_loss / beta + (1.0 - r) * negative_loss / beta
    policy_loss = (policy_loss_per_sample * adv_clip_max).mean()

    # ===== Step 6: KL 正则化 =====
    # L_KL = mean((v_θ - v_ref)²)
    # 防止当前策略偏离 reference model 太远, 保证训练稳定性
    kl_loss = ((v_theta - v_ref) ** 2).mean()

    # ===== Step 7: 总损失 =====
    # L_total = L_policy + β_kl * L_KL
    total_loss = policy_loss + beta_kl * kl_loss

    return total_loss, {
        "policy_loss": policy_loss.item(),
        "kl_loss": kl_loss.item(),
        "positive_loss": positive_loss.mean().item(),
        "negative_loss": negative_loss.mean().item(),
        "r_mean": r.mean().item(),
        "total_loss": total_loss.item(),
    }

## 完整训练循环 Demo

DiffusionNFT 的在线强化学习迭代分为 **四个阶段**：

| 阶段 | 名称 | 核心操作 |
|------|------|---------|
| Phase 1 | 数据采集 | 用 old policy 生成图像，外部奖励函数打分 |
| Phase 2 | Advantage 计算 | Per-Prompt 组内归一化 |
| Phase 3 | 前向过程训练 | 在加噪过程上计算 DiffusionNFT Loss 并更新参数 |
| Phase 4 | Old Policy 更新 | EMA 软更新 old policy |

### 初始化

创建**三个网络**：
- $v_\theta$：当前可训练策略
- $v_{\text{old}}$：old policy（EMA 跟踪 $v_\theta$，用于数据采样和速度场参考）
- $v_{\text{ref}}$：reference model（冻结，用于 KL 正则化）

In [ ]:
def run_demo():
    """DiffusionNFT 完整训练循环 Demo"""
    print("=" * 70)
    print("  DiffusionNFT 最小 Demo")
    print("  论文: Online Diffusion Reinforcement with Forward Process")
    print("  ICLR 2026 Oral, NVIDIA")
    print("=" * 70)

    # ---- 初始化三个网络 ----
    # 1. v_θ: 当前可训练策略 (参数会被更新)
    # 2. v_old: old policy (EMA 跟踪当前策略, 用于数据采集和速度场参考)
    # 3. v_ref: reference model (冻结不训练, 用于 KL 正则化约束)
    v_theta = SimpleVelocityNet().to(device)
    v_old = deepcopy(v_theta)     # old policy, 初始化为当前策略的拷贝
    v_ref = deepcopy(v_theta)     # reference model, 冻结
    for p in v_ref.parameters():
        p.requires_grad = False   # 冻结参考模型的参数

    # AdamW 优化器 (仅优化 v_theta)
    optimizer = torch.optim.AdamW(v_theta.parameters(), lr=LR)

    # Per-Prompt 奖励统计追踪器
    stat_tracker = PerPromptStatTracker(global_std=True)

In [ ]:
    # ---- 模拟 prompts (实际中是文本编码后的嵌入) ----
    prompts = ["a cat sitting on a chair", "a dog running in a park",
               "a cat sitting on a chair", "a dog running in a park"]
    cond_dim = 32
    prompt_embeds = torch.randn(len(prompts), cond_dim)    # 模拟 text embedding

    global_step = 0

    for epoch in range(3):
        print(f"\n{'='*50}")
        print(f"  Epoch {epoch}")
        print(f"{'='*50}")

### Phase 1：数据采集（Rollout）

用 old policy 生成图像样本，然后用外部奖励函数打分。

**DiffusionNFT 的关键优势**：在数据收集阶段可以使用**任意黑盒高阶 ODE 采样器**（如 DPM-Solver++），仅需 20~40 步即可生成高质量样本。训练时只需要最终的干净图像 $x_0$ 和对应的 reward，不需要存储完整的采样轨迹。

本 demo 中用随机数据模拟这个过程。

In [ ]:
        # ============================================================
        # Phase 1: 数据采集 (Sampling / Rollout)
        # ============================================================
        print("\n[Phase 1] 数据采集: 用 old policy 生成图像并计算奖励")

        # 模拟 "干净 latent" x_0
        # 实际中: 用 old policy 通过 ODE solver 采样得到完整图像
        # 这里简化: 直接随机生成, 模拟已经收集好的数据
        x0_collected = torch.randn(BATCH_SIZE, LATENT_DIM, LATENT_HW, LATENT_HW)

        # 简化处理: 直接用 latent 当作 "生成的图像"
        images = x0_collected

        # 调用奖励函数打分 (实际中会调用 GenEval/PickScore/HPSv2 等)
        rewards = mock_reward_fn(images).numpy()
        print(f"  奖励: {rewards.round(3)}")

### Phase 2：计算 Per-Prompt Advantage

对同一 prompt 下的样本做**组内归一化**（延续 GRPO 的精髓）：

$$\text{advantage} = \frac{\text{reward} - \text{per\_prompt\_mean}}{\text{global\_std}}$$

- 减去组内均值：以组内平均水平作为 baseline，衡量每个样本的相对好坏
- 除以标准差：归一化量纲，使不同 prompt 的 advantage 可比
- 同一个样本在不同时间步 $t$ 共享相同的 advantage

In [ ]:
        # ============================================================
        # Phase 2: 计算 Per-Prompt Advantage (组内归一化)
        # ============================================================
        print("\n[Phase 2] 计算 Per-Prompt Advantage")

        # 用 PerPromptStatTracker 计算每个样本的 advantage
        # advantage = (reward - per_prompt_mean) / global_std
        advantages = stat_tracker.update(prompts, rewards)
        print(f"  Advantage: {advantages.round(3)}")

        # 将 advantage 扩展为 [B, num_timesteps]
        # 原代码的做法: 同一个样本在不同时间步共享相同的 advantage
        advantages_tensor = torch.tensor(advantages, dtype=torch.float32).unsqueeze(1)
        advantages_tensor = advantages_tensor.repeat(1, NUM_TIMESTEPS)    # [B, T]

### Phase 3：前向过程训练（DiffusionNFT 核心）

这是 DiffusionNFT 最核心的训练步骤。对每个时间步 $t$：

1. **前向加噪**：$x_t = (1-t) \cdot x_0 + t \cdot \epsilon$，在干净样本上叠加噪声
2. **三路速度场预测**：分别用 $v_\theta$（当前策略）、$v_{\text{old}}$（旧策略）、$v_{\text{ref}}$（参考模型）预测速度
3. **计算 DiffusionNFT Loss**：包含正/负样本速度场混合、自适应加权 MSE、KL 正则化
4. **反向传播**：梯度裁剪后更新 $v_\theta$ 的参数

**关键设计**：只需要最终的干净图像 $x_0$，不需要完整的采样轨迹。中间状态 $x_t$ 是在训练时通过加噪公式在线生成的。

In [ ]:
        # ============================================================
        # Phase 3: 前向过程训练 (DiffusionNFT 核心)
        # ============================================================
        print("\n[Phase 3] 前向过程训练 (DiffusionNFT Loss)")

        # 随机采样时间步 (原代码从 [0, 1000] 采样, 这里简化)
        timesteps_all = torch.randint(1, 1000, (BATCH_SIZE, NUM_TIMESTEPS))

        for t_idx in range(NUM_TIMESTEPS):
            # 归一化时间步到 [0, 1]
            t = timesteps_all[:, t_idx].float() / 1000.0    # [B]
            t_expanded = t.view(-1, 1, 1, 1)                 # [B, 1, 1, 1] 便于广播

            # ===== Flow Matching 前向加噪过程 =====
            # 公式: x_t = (1-t) * x_0 + t * ε,  ε ~ N(0, I)
            # t=0 时 x_t = x_0 (干净), t=1 时 x_t ≈ ε (纯噪声)
            noise = torch.randn_like(x0_collected)
            x_t = (1 - t_expanded) * x0_collected + t_expanded * noise

            # ===== 三个速度场预测 =====

            # (a) Old policy 的速度场 (冻结, 不追踪梯度)
            with torch.no_grad():
                old_pred = v_old(x_t, t, prompt_embeds).detach()

            # (b) 当前策略的速度场 (可训练, 追踪梯度)
            theta_pred = v_theta(x_t, t, prompt_embeds)

            # (c) Reference model 的速度场 (冻结, 用于 KL 正则化)
            with torch.no_grad():
                ref_pred = v_ref(x_t, t, prompt_embeds).detach()

            # ===== 取出当前时间步的 advantage =====
            adv_for_step = advantages_tensor[:, t_idx]    # [B]

            # ===== 计算 DiffusionNFT Loss =====
            loss, loss_dict = diffusion_nft_loss(
                v_theta=theta_pred,
                v_old=old_pred,
                v_ref=ref_pred,
                x0=x0_collected,
                x_t=x_t,
                t_expanded=t_expanded,
                advantages=adv_for_step,
            )

            # ===== 反向传播 & 参数更新 =====
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(v_theta.parameters(), max_norm=1.0)  # 梯度裁剪
            optimizer.step()

            print(f"  t={t_idx}: loss={loss_dict['total_loss']:.4f}  "
                  f"policy={loss_dict['policy_loss']:.4f}  "
                  f"kl={loss_dict['kl_loss']:.6f}  "
                  f"r_mean={loss_dict['r_mean']:.3f}")

            global_step += 1

### Phase 4：更新 Old Policy（EMA 软更新）

每个 epoch 结束后，用 **EMA（指数移动平均）** 更新 old policy：

$$\theta^{\text{old}} \leftarrow \eta \cdot \theta^{\text{old}} + (1 - \eta) \cdot \theta$$

- $\eta$ 由 `return_decay()` 动态计算，随训练步数逐渐增大
- 这使得 old policy 缓慢跟踪当前策略的变化，避免训练不稳定

In [ ]:
        # ============================================================
        # Phase 4: 更新 Old Policy (EMA 软更新)
        # ============================================================
        print("\n[Phase 4] 更新 Old Policy (EMA)")

        # 根据当前步数计算 EMA 衰减系数
        decay = return_decay(global_step, DECAY_TYPE)
        print(f"  Decay = {decay:.4f}")

        # EMA 更新公式: θ_old = decay * θ_old + (1 - decay) * θ_current
        # decay 越大, old policy 变化越慢, 训练越稳定但响应越迟钝
        with torch.no_grad():
            for src_param, tgt_param in zip(v_theta.parameters(), v_old.parameters()):
                tgt_param.data.copy_(
                    tgt_param.data * decay + src_param.data.clone() * (1.0 - decay)
                )

        # 验证: 计算 old policy 和当前策略的参数距离
        param_diff = sum(
            (p_old - p_cur).pow(2).sum().item()
            for p_old, p_cur in zip(v_old.parameters(), v_theta.parameters())
        )
        print(f"  ||θ_old - θ_current||² = {param_diff:.6f}")

        # 清空 advantage 统计缓存 (每个 epoch 重新统计)
        stat_tracker.stats.clear()

## 总结

Demo 结束后，打印 DiffusionNFT 的所有核心公式和关键洞察的总结。

In [ ]:
    # ============================================================
    # 总结: 打印 DiffusionNFT 核心要点
    # ============================================================
    print("\n" + "=" * 70)
    print("  Demo 完成! 以下是 DiffusionNFT 的核心要点总结:")
    print("=" * 70)
    print("""
    1. [前向过程] Flow Matching:
       x_t = (1-t) * x_0 + t * ε

    2. [速度场] 模型预测速度 v_θ(x_t, t), 恢复 x_0:
       x̂_0 = x_t - t * v_θ(x_t, t)

    3. [核心损失] DiffusionNFT Loss:
       r = clamp(adv/(2*max) + 0.5, 0, 1)          # 奖励权重
       v_pos = β*v_θ + (1-β)*v_old                  # 正样本速度场
       v_neg = (1+β)*v_old - β*v_θ                  # 隐式负样本速度场
       x̂_0_pos = x_t - t*v_pos                      # 正样本 x_0 预测
       x̂_0_neg = x_t - t*v_neg                      # 负样本 x_0 预测
       w = mean(|x̂_0 - x_0|).detach()               # 自适应权重
       L_pos = mean((x̂_0_pos - x_0)²/w)             # 正样本损失
       L_neg = mean((x̂_0_neg - x_0)²/w)             # 负样本损失
       L_policy = r*L_pos/β + (1-r)*L_neg/β          # 策略损失
       L_KL = mean((v_θ - v_ref)²)                   # KL 正则化
       L_total = L_policy + β_kl * L_KL              # 总损失

    4. [Old Policy] EMA 更新:
       θ_old = decay * θ_old + (1-decay) * θ_current

    5. [Advantage] Per-Prompt 归一化:
       advantage = (reward - per_prompt_mean) / global_std

    关键洞察:
    - 在前向过程上做 RL, 只需干净图像 x_0, 不需完整采样轨迹
    - β 控制正/负样本的混合强度
    - r 将 reward 映射为 [0,1] 的权重, 决定正/负损失的占比
    - v_old 提供稳定的参考点, 避免训练震荡
    - KL 正则化防止策略偏离 reference model
    """)

In [ ]:
run_demo()